# 01 · Сбор данных

Этот ноутбук собирает сырые данные проекта.

- Часть 1: почасовой PM2.5 станции из OpenAQ (цепочка «локация → сенсор → измерения»).
- Часть 2: почасовая погода и модельный PM2.5 (CAMS) из Open-Meteo.

Запускать сверху вниз.

## Шаг 1. Импорты

Подключаем pandas и наш модуль `src/collect.py`. Ноутбук запускается из папки `notebooks/`, поэтому добавляем корень проекта в путь.

In [1]:
import os
import sys

import pandas as pd

sys.path.append("..")
from src import collect

RAW_PATH = "../data/raw/openaq_pm25.csv"

## Шаг 2. Что выгружаем

Сенсор PM2.5 и период взяты из задачи 1.1. Время в сырых данных — UTC, как его отдаёт API. Пауза 1 с после каждой непустой страницы; всего 47 запросов, это меньше лимита 60 в минуту.

In [2]:
print("sensor_id:", collect.SENSOR_ID)
print("from (UTC):", collect.PERIOD_FROM_UTC)
print("to (UTC):", collect.PERIOD_TO_UTC)
start = pd.Timestamp(collect.PERIOD_FROM_UTC)
end = pd.Timestamp(collect.PERIOD_TO_UTC)
max_hours = int((end - start) / pd.Timedelta(hours=1))
print("max hours in period:", max_hours)

sensor_id: 25903
from (UTC): 2020-04-09T16:00:00Z
to (UTC): 2025-11-14T11:00:00Z
max hours in period: 49075


## Шаг 3. Выгрузка один раз

Качаем страницами по 1000 строк до пустой страницы, по годовым окнам. Между запросами пауза 1 с (лимит API — 60 запросов в минуту), при ответе 429 есть повтор. Сырой файл не перезаписываем: если он уже есть, API не вызываем.

In [3]:
if os.path.exists(RAW_PATH):
    print("Файл уже есть, API не вызываем:", RAW_PATH)
else:
    pm25_download = collect.download_pm25()
    collect.save_raw_csv(pm25_download, RAW_PATH)
    print("Сохранено строк:", len(pm25_download))

Файл уже есть, API не вызываем: ../data/raw/openaq_pm25.csv


## Шаг 4. Читаем сырой файл

Читаем CSV с диска и смотрим размер и первые строки.

In [4]:
pm25_raw = pd.read_csv(RAW_PATH)
print(pm25_raw.shape)
pm25_raw.head()

(37823, 4)


,datetime_from_utc,datetime_to_utc,pm25,has_flags
0,2020-04-09T16:00:00Z,2020-04-09T17:00:00Z,6.0,False
1,2020-04-09T17:00:00Z,2020-04-09T18:00:00Z,10.0,False
2,2020-04-09T18:00:00Z,2020-04-09T19:00:00Z,8.0,False
3,2020-04-09T19:00:00Z,2020-04-09T20:00:00Z,6.0,False
4,2020-04-09T20:00:00Z,2020-04-09T21:00:00Z,4.0,False


## Шаг 5. Инспекция

Смотрим типы колонок и пропуски. Ничего не исправляем: чистка — это Этап 2. Колонка `has_flags` — пометка станции о проблеме с измерением; смотрим, сколько строк помечено и все ли пропуски PM2.5 помечены.

In [5]:
print(pm25_raw.dtypes)
print()
print(pm25_raw.isna().sum())
print()
print(pm25_raw["has_flags"].value_counts())
print()
print(pm25_raw.loc[pm25_raw["pm25"].isna(), "has_flags"].value_counts())

datetime_from_utc        str
datetime_to_utc          str
pm25                 float64
has_flags               bool
dtype: object

datetime_from_utc     0
datetime_to_utc       0
pm25                 66
has_flags             0
dtype: int64

has_flags
False    37710
True       113
Name: count, dtype: int64

has_flags
True    66
Name: count, dtype: int64


## Шаг 6. Время

Парсим обе метки явно (UTC, формат задан) и считаем NaT: так видно, не сломались ли даты. `datetime_from_utc` — начало часа, `datetime_to_utc` — конец часа. Период сенсора из задачи 1.1 (17:00 … 11:00) задан по концу окна, поэтому первая строка начинается в 16:00. Длина каждого окна должна быть один час.

In [6]:
time_format = "%Y-%m-%dT%H:%M:%SZ"
times = pd.to_datetime(pm25_raw["datetime_from_utc"], utc=True, format=time_format, errors="coerce")
times_to = pd.to_datetime(pm25_raw["datetime_to_utc"], utc=True, format=time_format, errors="coerce")
print("NaT from:", times.isna().sum())
print("NaT to:", times_to.isna().sum())
print("first:", times.min())
print("last:", times.max())
print()
print((times_to - times).value_counts())

NaT from: 0
NaT to: 0
first: 2020-04-09 16:00:00+00:00
last: 2025-11-14 10:00:00+00:00

0 days 01:00:00    37823
Name: count, dtype: int64


## Шаг 7. Полнота ряда

Сравниваем число строк с максимумом часов в периоде: разница — пропущенные часы (дыры станции). Также считаем дубли меток и строки по годам.

In [7]:
unique_hours = times.nunique()
missing_hours = max_hours - unique_hours
missing_share = missing_hours / max_hours * 100
print("max hours in period:", max_hours)
print("unique hours:", unique_hours)
print("missing hours:", missing_hours)
print("missing share, %:", round(missing_share, 1))
print("duplicated timestamps:", times.duplicated().sum())
print()
print(times.dt.year.value_counts().sort_index())

max hours in period: 49075
unique hours: 37823
missing hours: 11252
missing share, %: 22.9
duplicated timestamps: 0

datetime_from_utc
2020    4735
2021    5124
2022    4728
2023    7344
2024    8318
2025    7574
Name: count, dtype: int64


## Шаг 8. Значения PM2.5

Смотрим распределение значений (µg/m³) и считаем отрицательные значения и нули.

In [8]:
print(pm25_raw["pm25"].describe())
print("negative:", (pm25_raw["pm25"] < 0).sum())
print("zeros:", (pm25_raw["pm25"] == 0).sum())

count    37757.000000
mean        29.547236
std         37.862882
min          0.000000
25%          9.000000
50%         15.000000
75%         31.000000
max        421.000000
Name: pm25, dtype: float64
negative: 0
zeros: 90


## Итог части 1

Сырой файл прочитан и осмотрен без изменений. В Этап 2 передаём то, что видно в выводе ячеек выше:

- пропущенные часы: в ряду нет строк для части часов периода (число и доля — в шаге 7);
- пропуски значения PM2.5 (NaN): строки есть, но значения нет; все они помечены в `has_flags`;
- нули в PM2.5 (число — в шаге 8) и строки с `has_flags = True` (число — в шаге 5): решить, считать ли их настоящими измерениями;
- дублей меток времени и отрицательных значений нет (шаги 7 и 8).

Решать, что с этим делать, будем при очистке.

# Часть 2. Open-Meteo: погода и CAMS

Берём два источника Open-Meteo. Ключ не нужен, координаты — станции из задачи 1.1.

- Историческая погода: температура, влажность, осадки, давление, ветер.
- CAMS — это **модельный** PM2.5 (расчёт на сетке), а не измерения; он нужен для гипотезы H4.

Время запрашиваем в UTC, как у OpenAQ. Проба показала: с `timezone=Asia/Almaty` API отдаёт один сдвиг +5 на весь период (`utc_offset_seconds=18000`), хотя до 1 марта 2024 Алматы жила по UTC+6, — до марта 2024 местное время было бы сдвинуто на час. UTC такого сдвига не даёт и совпадает с OpenAQ. В Asia/Almaty переводим на этапе 2.

Лицензия данных: «Weather data by Open-Meteo.com» (CC BY 4.0). Contains modified Copernicus Atmosphere Monitoring Service information.

## Шаг 9. Что выгружаем

Задаём пути сырых файлов и печатаем параметры запроса: координаты, даты, список переменных и ожидаемое число часов погоды. Так видно, с чем потом сверять выгрузку.

In [9]:
WEATHER_PATH = "../data/raw/weather.csv"
CAMS_PATH = "../data/raw/cams_aq.csv"

print("Широта:", collect.LATITUDE)
print("Долгота:", collect.LONGITUDE)
print("Погода: с", collect.METEO_START_DATE, "по", collect.METEO_END_DATE)
print("CAMS: с", collect.CAMS_START_DATE, "по", collect.METEO_END_DATE)
print("Переменные погоды:", list(collect.WEATHER_UNITS))

weather_start = pd.Timestamp(collect.METEO_START_DATE)
weather_end = pd.Timestamp(collect.METEO_END_DATE) + pd.Timedelta(hours=23)
expected_hours = (weather_end - weather_start) / pd.Timedelta(hours=1) + 1
print("Ожидаем часов погоды:", int(expected_hours))

Широта: 43.252855
Долгота: 76.93118
Погода: с 2020-04-09 по 2025-11-14
CAMS: с 2022-08-04 по 2025-11-14
Переменные погоды: ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'pressure_msl', 'wind_speed_10m', 'wind_direction_10m']
Ожидаем часов погоды: 49104


## Шаг 10. Погода: выгрузка один раз

Один запрос за весь период, пагинации нет, ветер в м/с. Сырой файл не перезаписываем: если он уже есть, в API не ходим.

In [10]:
if os.path.exists(WEATHER_PATH):
    print("Файл уже есть, в API не ходим:", WEATHER_PATH)
else:
    weather_download = collect.download_weather()
    collect.save_raw_csv(weather_download, WEATHER_PATH)
    print("Сохранено строк:", len(weather_download))

Файл уже есть, в API не ходим: ../data/raw/weather.csv


## Шаг 11. CAMS: выгрузка один раз

Начинаем с `CAMS_START_DATE`, а не с 2020 года: раньше этой даты API возвращает пустые значения. Файл тоже пишем один раз.

In [11]:
if os.path.exists(CAMS_PATH):
    print("Файл уже есть, в API не ходим:", CAMS_PATH)
else:
    cams_download = collect.download_cams()
    collect.save_raw_csv(cams_download, CAMS_PATH)
    print("Сохранено строк:", len(cams_download))

Файл уже есть, в API не ходим: ../data/raw/cams_aq.csv


## Шаг 12. Читаем сырые файлы

Читаем оба файла как есть, ничего не исправляем: сначала смотрим размер и первые строки. Так мы видим, что именно лежит в сырых данных, до любой очистки.

In [12]:
weather_raw = pd.read_csv(WEATHER_PATH)
cams_raw = pd.read_csv(CAMS_PATH)

print(weather_raw.shape)
print(cams_raw.shape)
weather_raw.head()

(49104, 7)
(28776, 2)


,time_utc,temp_c,humidity,precip,pressure,wind_speed,wind_dir
0,2020-04-09T00:00,2.4,94,0.5,1026.8,2.33,245
1,2020-04-09T01:00,2.2,99,0.5,1027.5,0.72,326
2,2020-04-09T02:00,2.5,94,0.3,1028.1,1.14,232
3,2020-04-09T03:00,2.9,91,0.2,1028.6,1.48,298
4,2020-04-09T04:00,2.8,93,0.4,1030.2,1.92,261


## Шаг 13. Время

Метки в файлах без секунд и без `Z` (формат `2024-02-28T00:00`), поэтому `utc=True` и `format=` указываем явно. У Open-Meteo полная сетка часов: пропуск — это пустое значение, а не пропавшая строка (в OpenAQ было наоборот). Считаем NaT, первую и последнюю метки, дубли и шаг между метками.

In [13]:
weather_times = pd.to_datetime(weather_raw["time_utc"], utc=True, format=collect.METEO_TIME_FORMAT, errors="coerce")
cams_times = pd.to_datetime(cams_raw["time_utc"], utc=True, format=collect.METEO_TIME_FORMAT, errors="coerce")

for name, source_times in [("Погода", weather_times), ("CAMS", cams_times)]:
    print(name)
    print("  NaT:", source_times.isna().sum())
    print("  Первая метка:", source_times.iloc[0])
    print("  Последняя метка:", source_times.iloc[-1])
    print("  Дублей:", source_times.duplicated().sum())
    print("  Шаг между метками:")
    print(source_times.diff().value_counts())

Погода
  NaT: 0
  Первая метка: 2020-04-09 00:00:00+00:00
  Последняя метка: 2025-11-14 23:00:00+00:00
  Дублей: 0
  Шаг между метками:
time_utc
0 days 01:00:00    49103
Name: count, dtype: int64
CAMS
  NaT: 0
  Первая метка: 2022-08-04 00:00:00+00:00
  Последняя метка: 2025-11-14 23:00:00+00:00
  Дублей: 0
  Шаг между метками:
time_utc
0 days 01:00:00    28775
Name: count, dtype: int64


## Шаг 14. Погода: пропуски и диапазоны

Смотрим типы, число пропусков и описательную статистику. Что ожидаем по смыслу: влажность 0–100 %, ветер не меньше 0 м/с, направление ветра 0–360°, осадки не меньше 0 мм. Если что-то выбивается — только фиксируем, решение принимаем на этапе очистки.

In [14]:
print(weather_raw.dtypes)
print()
print(weather_raw.isna().sum())
weather_raw.describe()

time_utc          str
temp_c        float64
humidity        int64
precip        float64
pressure      float64
wind_speed    float64
wind_dir        int64
dtype: object

time_utc      0
temp_c        0
humidity      0
precip        0
pressure      0
wind_speed    0
wind_dir      0
dtype: int64


,temp_c,humidity,precip,pressure,wind_speed,wind_dir
count,49104.000000,49104.000000,49104.000000,49104.000000,49104.000000,49104.00000
mean,11.873153,55.073884,0.059883,1018.549723,1.563412,181.14897
std,11.913288,21.377138,0.323194,9.978545,0.910536,106.37092
min,-28.400000,7.000000,0.000000,994.600000,0.000000,1.00000
25%,2.400000,38.000000,0.000000,1010.800000,0.920000,101.00000
50%,13.000000,53.000000,0.000000,1017.500000,1.430000,169.00000
75%,21.300000,72.000000,0.000000,1025.400000,2.020000,270.00000
max,39.100000,100.000000,10.600000,1058.900000,10.740000,360.00000


## Шаг 15. CAMS: пропуски и значения

То же для модельного PM2.5: пропуски, описательная статистика, число отрицательных значений (концентрация не может быть меньше нуля) и число строк по годам.

In [15]:
print(cams_raw.isna().sum())
print()
print(cams_raw["cams_pm25"].describe())
print()
print("Отрицательных значений:", (cams_raw["cams_pm25"] < 0).sum())
print()
print(cams_times.dt.year.value_counts().sort_index())

time_utc     0
cams_pm25    0
dtype: int64

count    28776.000000
mean        22.812132
std         14.400580
min          0.400000
25%         13.100000
50%         18.600000
75%         28.300000
max        153.700000
Name: cams_pm25, dtype: float64

Отрицательных значений: 0

time_utc
2022    3600
2023    8760
2024    8784
2025    7632
Name: count, dtype: int64


## Итог части 2

Сырые файлы Open-Meteo выгружены и проверены: `data/raw/weather.csv` (погода) и `data/raw/cams_aq.csv` (модельный PM2.5 CAMS). Файлы не менялись. В задачу 2.1 передаём то, что видно в выводе шагов 13–15: пропусков, дублей и значений вне ожидаемых диапазонов там нет. Ряд CAMS начинается 2022-08-04, поэтому окно сравнения для H4 короче периода OpenAQ.

Напоминание: все метки времени в UTC. В `Asia/Almaty` переводим один раз на этапе 2.